# 模型请求执行 Tool，就代表它拥有执行权限吗？

## V0.6 Capability Core / Enforcement

这个 lab 让同一个 tool call 分别经过 allowed agent 和 denied agent。

**Core:** Model proposal != Kernel authority.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Register one structured-capability Tool

In [ ]:
import asyncio
from collections.abc import Mapping

from agentkernel import (
    Agent, CapabilityGrant, ErrorCode, Session, TOOL_EXECUTE_ACTION,
    ToolCall, ToolDefinition, ToolExecutionContext, ToolRegistry, ToolSchema,
)
from agentkernel.protocol import JsonValue

async def add(arguments: Mapping[str, JsonValue], _context: ToolExecutionContext) -> JsonValue:
    return int(arguments["left"]) + int(arguments["right"])

registry = ToolRegistry()
registry.register(ToolDefinition(
    schema=ToolSchema("math.add", "Add two integers.", {"type": "object"}),
    handler=add,
    required_action=TOOL_EXECUTE_ACTION,
    required_resource="tool://math.add",
))
allowed_agent = Agent.create(
    agent_id="agent-allowed",
    session=Session("session-allowed"),
    capability_grants=(CapabilityGrant("agent-allowed", TOOL_EXECUTE_ACTION, "tool://math.add"),),
)
denied_agent = Agent.create(agent_id="agent-denied", session=Session("session-denied"))
print_table(grant_rows(allowed_agent.control.capability_grants))

## 2. Model-visible schemas are filtered by authority

In [ ]:
print_table([
    {"agent": "agent-allowed", "visible_tools": [schema.name for schema in registry.model_schemas(allowed_agent.control)]},
    {"agent": "agent-denied", "visible_tools": [schema.name for schema in registry.model_schemas(denied_agent.control)]},
])

## 3. Execution is checked again at the boundary

In [ ]:
call = ToolCall("call-add-1", "math.add", {"left": 20, "right": 22})
allowed = asyncio.run(registry.execute(call, allowed_agent.control))
denied = asyncio.run(registry.execute(call, denied_agent.control))
print_table([
    {"agent": "agent-allowed", "ok": allowed.ok, "result_or_error": allowed.output},
    {"agent": "agent-denied", "ok": denied.ok, "result_or_error": denied.error.code.value if denied.error else "none"},
])
assert denied.error is not None and denied.error.code is ErrorCode.EACCES
trajectory("Model proposal", "ToolRegistry authorization", "ALLOW or EACCES", "Execute only if allowed")

## Invariant

The model can propose a call, but the Kernel decides whether that call crosses the Tool boundary. `EACCES` means “permission denied.”

## WHAT THIS DEMONSTRATES / 本实验验证什么

- Capability grants affect model-visible tools.
- Execution is checked again even if a call is manually constructed.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not implement RBAC, IAM, namespace, or revocation.
- It does not prove production sandbox security.
- It does not use a real model provider.